In [ ]:
import numpy as np 
import pandas as pd
import random
from base64 import b64encode
from IPython.display import HTML
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import gc
import os
import csv
from tqdm.auto import tqdm
from typing import List, Tuple


from huggingface_hub import notebook_login
from transformers import CLIPProcessor, CLIPModel
from transformers import ViTImageProcessor, ViTModel, ViTFeatureExtractor
from transformers import AutoImageProcessor, ResNetModel
from diffusers import AutoencoderKL, LMSDiscreteScheduler, UNet2DConditionModel, StableDiffusionPipeline, StableDiffusionXLPipeline  
from diffusers import EulerDiscreteScheduler, DDIMScheduler, EulerAncestralDiscreteScheduler

import torch
from torch import autocast
from torchvision import transforms
from torchvision.transforms import ToPILImage
import torch.nn.functional as F
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from skimage.metrics import structural_similarity as ssim
from torch.nn.functional import cosine_similarity
from skimage.transform import resize

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)

In [ ]:
activitats = ["sleeping", "napping", "falling sleep", "waking up", "relaxing", "being sick in bed", "in a medical bed"] #etc

In [ ]:
def generate_quadruplets_for_activities(activities):
    quadruplets = []

    for activity in activities:
        activity = activity.lower()

        neutral = f"An ultra realistic portrait photo of a person {activity}"
        young = f"An ultra realistic portrait photo of a 25 years-old young person {activity}"
        middle = f"An ultra realistic portrait photo of a 45 year-old person middle-aged person {activity}"
        older = f"An ultra realistic portrait photo of a 75 years-old older person {activity}"

        quadruplets.append((neutral, young, middle, older))

    return quadruplets

quadruplets = generate_quadruplets_for_activities(activitats)
quadruplets

In [ ]:
csv_path = "/Quadruplets_Activities.csv"
desktop_path = os.path.join(os.path.expanduser("~"), "Documents", "Quadruplets_Activities.csv")

total_activities = len(activities)

with open(desktop_path, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(["Activity"])
    for activity in activities:
        writer.writerow([activity.lower()])

total_activities, desktop_path

# Split activities into 4 chunks
chunk_size = len(activities) // 4
activities_1 = activities[:chunk_size]
activities_2 = activities[chunk_size:chunk_size*2]
activities_3 = activities[chunk_size*2:chunk_size*3]
activities_4 = activities[chunk_size*3:]

In [ ]:
torch.cuda.empty_cache()
gc.collect()

class Evaluator:
    def __init__(self, pipe, clip_model, clip_processor, scheduler_name: str = "default"):
        self.pipe = pipe
        self.clip_model = clip_model
        self.clip_processor = clip_processor
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # Initialize DINO-s16
        self.dino_s16_model = ViTImageProcessor.from_pretrained("facebook/dino-vits16", add_pooling_layer=False)
        self.dino_s16_processor = ViTModel.from_pretrained("facebook/dino-vits16", add_pooling_layer=False).to(self.device)

        # Initialize DINO-b8
        self.dino_b8_model = ViTImageProcessor.from_pretrained("facebook/dino-vits8",add_pooling_layer=False)
        self.dino_b8_processor = ViTModel.from_pretrained("facebook/dino-vits8",add_pooling_layer=False).to(self.device)

        # Initialize ResNet-50 from Hugging Face
        self.resnet_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
        self.resnet_model = ResNetModel.from_pretrained("microsoft/resnet-50").to(self.device)

        self.resnet_torch_transform = Compose([
            Resize((224, 224)),
            ToTensor(),
            Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        if scheduler_name != "default":
            self.set_scheduler(scheduler_name)

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.ConvertImageDtype(torch.float32),
            transforms.Normalize(
                mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711]
            )
        ])

    # Scheduler
    def set_scheduler(self, scheduler_name: str):
        if scheduler_name == "euler":
            self.pipe.scheduler = EulerDiscreteScheduler.from_config(self.pipe.scheduler.config)
        elif scheduler_name == "euler_ancestral":
            self.pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(self.pipe.scheduler.config)
        elif scheduler_name == "ddim":
            self.pipe.scheduler = DDIMScheduler.from_config(self.pipe.scheduler.config)
        else:
            raise ValueError(f"Scheduler '{scheduler_name}' not recognized. Valid options: 'euler', 'euler_ancestral', 'ddim'.")

    # Generate Images
    def generate_images(self, prompt: str, seed: int = None) -> Image:
        if seed is not None:
            generator = torch.Generator(device=self.device).manual_seed(seed)
        else:
            generator = None

        image = self.pipe(prompt,
                          num_inference_steps=20,
                          guidance_scale=7.5,
                          height=768,
                          width=768,
                          negative_prompt="painting, cartoon, anime, render, artwork, 3d render, unrealistic, vector art,cgi, render, deformed hands, extra limbs, bad anatomy, abstract, digital art, drawing, illustration, sketch, doodle, graphic design, black and white, B/W, sepia",
                          generator=generator).images[0]
        return image

    # CLIP Embeddings
    def get_clip_text_embeddings(self, prompts: Tuple[str, str, str, str]):
        inputs = self.clip_processor(text=list(prompts), return_tensors="pt", padding=True).to(self.device)
        return self.clip_model.get_text_features(**inputs)

    def get_clip_image_embeddings(self, image: Image):
        inputs = self.clip_processor(images=image, return_tensors="pt").to(self.device)
        return self.clip_model.get_image_features(**inputs)

    # Latent Representations with VAE
    def get_latent_representation_from_vae(self, image: Image):
        try:
            image = image.convert('RGB')
            image_tensor = self.transform(image).unsqueeze(0).to(self.device)

            vae = self.pipe.vae.to_empty(device=self.device)
            vae = self.pipe.vae.to(self.device)

            with torch.no_grad():
                try:
                    latent_dist = vae.encode(image_tensor).latent_dist
                except Exception as e:
                    print(f"\nEncoding Method 1 Failed: {e}")

                    try:
                        resized_tensor = F.interpolate(image_tensor, size=(256, 256), mode='bilinear', align_corners=False)
                        latent_dist = vae.encode(resized_tensor).latent_dist
                    except Exception as e:
                        print(f"\nEncoding Method 2 Failed: {e}")
                        return None

                latents = latent_dist.sample()
                latents = latents * vae.config.scaling_factor
                latents = latents.flatten().float()
                latents = latents / torch.norm(latents)

                return latents

        except Exception as e:
            print(f"Comprehensive error in VAE encoding: {e}")
            import traceback
            traceback.print_exc()
            return None

    # Cosine Similarity
    def compute_cosine_similarity(self, vec1, vec2):
        if vec1 is None or vec2 is None:
            print("One of the vectors is None")
            return None

        try:
            vec1 = vec1.float().to(self.device)
            vec2 = vec2.float().to(self.device)

            vec1 = vec1.flatten()
            vec2 = vec2.flatten()

            min_length = min(len(vec1), len(vec2))
            vec1 = vec1[:min_length]
            vec2 = vec2[:min_length]

            vec1 = vec1 / torch.norm(vec1)
            vec2 = vec2 / torch.norm(vec2)

            similarity = torch.nn.functional.cosine_similarity(vec1.unsqueeze(0), vec2.unsqueeze(0), dim=1)
            return similarity.item()

        except Exception as e:
            print(f"Cosine similarity calculation error: {e}")
            import traceback
            traceback.print_exc()
            return None

    # ResNet Similarity
    def compute_resnet_similarity(self, image1: Image, image2: Image) -> float:
        """
        Calculates similarity between two images using ResNet-50.
        Extracts features and computes cosine similarity.
        """
        try:
            image1 = image1.resize((224, 224))
            image2 = image2.resize((224, 224))

            inputs1 = self.resnet_processor(image1, return_tensors="pt").to(self.device)
            inputs2 = self.resnet_processor(image2, return_tensors="pt").to(self.device)

            with torch.no_grad():
                features1 = self.resnet_model(**inputs1).pooler_output
                features2 = self.resnet_model(**inputs2).pooler_output

            if features1 is None or features2 is None:
                print("Error: one of the vectors is None")
                return None

            features1 = F.normalize(features1, p=2, dim=-1)
            features2 = F.normalize(features2, p=2, dim=-1)

            similarity = torch.nn.functional.cosine_similarity(features1, features2).item()
            return similarity

        except Exception as e:
            print(f"Error calculating similarity with ResNet-50: {e}")
            return None

    # DINO Embeddings
    def get_dino_embeddings(self, image: Image):
        """
        Obtains DINO embeddings using only the s16 model (DINO-ViT).
        """
        try:
            feature_extractor = ViTFeatureExtractor.from_pretrained("facebook/dino-vits16", size=224)
            inputs = feature_extractor(images=image, return_tensors="pt").pixel_values.to(self.device)
            model = ViTModel.from_pretrained("facebook/dino-vits16", add_pooling_layer=False).to(self.device)

            with torch.no_grad():
                outputs = model(inputs).last_hidden_state.mean(dim=1)

            embeddings = outputs.flatten()
            embeddings /= embeddings.norm(p=2)
            return embeddings

        except Exception as e:
            print(f"Error generating DINO-s16 embeddings: {e}")
            return None

    # Calculate Split-Product using DINO-b8
    def compute_split_product(self, image1: Image, image2: Image) -> float:
        """
        Calculates split-product based on maximum patch similarity
        between two images using DINO-s16.
        """
        try:
            feature_extractor = ViTFeatureExtractor.from_pretrained("facebook/dino-vits16", size=224)
            inputs1 = feature_extractor(images=image1, return_tensors="pt").pixel_values.to(self.device)
            inputs2 = feature_extractor(images=image2, return_tensors="pt").pixel_values.to(self.device)

            model = ViTModel.from_pretrained("facebook/dino-vits16", add_pooling_layer=False).to(self.device)

            with torch.no_grad():
                outputs1 = model(inputs1).last_hidden_state
                outputs2 = model(inputs2).last_hidden_state

            patches1 = F.normalize(outputs1, p=2, dim=-1)
            patches2 = F.normalize(outputs2, p=2, dim=-1)

            similarities = torch.matmul(patches1, patches2.transpose(-1, -2))
            max_similarities = similarities.max(dim=-1).values
            split_product = max_similarities.mean().item()

            return split_product

        except Exception as e:
            print(f"Error calculating split-product: {e}")
            return None

    # Cache Functions
    def load_cached_image(self, filename: str, output_dir: str) -> Image.Image:
        """
        Attempts to load an image from cache if it already exists.
        """
        filepath = os.path.join(output_dir, filename)
        if os.path.exists(filepath):
            print(f"Loading cached image: {filepath}")
            return Image.open(filepath).convert("RGB")
        return None

    def cache_image(self, image: Image.Image, filename: str, output_dir: str):
        """
        Saves an image to the cache directory.
        """
        os.makedirs(output_dir, exist_ok=True)
        filepath = os.path.join(output_dir, filename)
        image.save(filepath)
        print(f"Image cached: {filepath}")

    def generate_and_cache_image(self, prompt: str, output_dir: str = "generated_images"):
        """
        Generates an image for a prompt, using cache if already generated.
        """
        filename = prompt.replace(" ", "_").lower() + ".png"

        cached_image = self.load_cached_image(filename, output_dir)
        if cached_image:
            return cached_image

        print(f"Generating image for prompt: '{prompt}'")
        generated_image = self.generate_images(prompt)
        self.cache_image(generated_image, filename, output_dir)
        return generated_image

    # Main Execution
    def evaluate_prompt_quadruplet_denoising_space(self, quadruplet: Tuple[str, str, str, str], seed: int = None, output_dir: str = "generated_images") -> tuple:
        """
        Generates images, calculates similarity metrics, and returns a dictionary of metrics and images.
        """
        neutral_prompt, young_prompt, middle_prompt, older_prompt = quadruplet

        neutral_image = self.generate_and_cache_image(neutral_prompt, output_dir)
        young_image = self.generate_and_cache_image(young_prompt, output_dir)
        middle_image = self.generate_and_cache_image(middle_prompt, output_dir)
        older_image = self.generate_and_cache_image(older_prompt, output_dir)

        images = {
            "neutral": neutral_image,
            "young": young_image,
            "middle": middle_image,
            "older": older_image,
        }

        clip_text_embeddings = self.get_clip_text_embeddings((neutral_prompt, young_prompt, middle_prompt, older_prompt))

        neutral_image_embedding = self.get_clip_image_embeddings(neutral_image)
        young_image_embedding = self.get_clip_image_embeddings(young_image)
        middle_image_embedding = self.get_clip_image_embeddings(middle_image)
        older_image_embedding = self.get_clip_image_embeddings(older_image)

        resnet_similarity_neutral_young = self.compute_resnet_similarity(neutral_image, young_image)
        resnet_similarity_neutral_middle = self.compute_resnet_similarity(neutral_image, middle_image)
        resnet_similarity_neutral_older = self.compute_resnet_similarity(neutral_image, older_image)

        neutral_dino_s16 = self.get_dino_embeddings(neutral_image)
        young_dino_s16 = self.get_dino_embeddings(young_image)
        middle_dino_s16 = self.get_dino_embeddings(middle_image)
        older_dino_s16 = self.get_dino_embeddings(older_image)

        neutral_latent = self.get_latent_representation_from_vae(neutral_image)
        young_latent = self.get_latent_representation_from_vae(young_image)
        middle_latent = self.get_latent_representation_from_vae(middle_image)
        older_latent = self.get_latent_representation_from_vae(older_image)

        split_product_neutral_young = self.compute_split_product(neutral_image, young_image)
        split_product_neutral_middle = self.compute_split_product(neutral_image, middle_image)
        split_product_neutral_older = self.compute_split_product(neutral_image, older_image)

        print("Neutral latent:", neutral_latent)
        print("young latent:", young_latent)
        print("middle latent:", middle_latent)
        print("older latent:", older_latent)

        metrics = {
            # CLIP Similarities
            "cosine_prompt_neutral_young": self.compute_cosine_similarity(clip_text_embeddings[0], clip_text_embeddings[1]),
            "cosine_prompt_neutral_middle": self.compute_cosine_similarity(clip_text_embeddings[0], clip_text_embeddings[2]),
            "cosine_prompt_neutral_older": self.compute_cosine_similarity(clip_text_embeddings[0], clip_text_embeddings[3]),
            "cosine_image_neutral_young": self.compute_cosine_similarity(neutral_image_embedding, young_image_embedding),
            "cosine_image_neutral_middle": self.compute_cosine_similarity(neutral_image_embedding, middle_image_embedding),
            "cosine_image_neutral_older": self.compute_cosine_similarity(neutral_image_embedding, older_image_embedding),

            # ResNet
            "resnet_similarity_neutral_young": resnet_similarity_neutral_young,
            "resnet_similarity_neutral_middle": resnet_similarity_neutral_middle,
            "resnet_similarity_neutral_older": resnet_similarity_neutral_older,

            # DINO-s16 Similarities
            "cosine_dino_s16_neutral_young": self.compute_cosine_similarity(neutral_dino_s16, young_dino_s16),
            "cosine_dino_s16_neutral_middle": self.compute_cosine_similarity(neutral_dino_s16, middle_dino_s16),
            "cosine_dino_s16_neutral_older": self.compute_cosine_similarity(neutral_dino_s16, older_dino_s16),

            # Split Product
            "split_product_young": split_product_neutral_young,
            "split_product_middle": split_product_neutral_middle,
            "split_product_older": split_product_neutral_older,
        }

        # Only add latent metrics if available
        if neutral_latent is not None and young_latent is not None:
            metrics["cosine_latent_neutral_young"] = self.compute_cosine_similarity(neutral_latent, young_latent)

        if neutral_latent is not None and middle_latent is not None:
            metrics["cosine_latent_neutral_middle"] = self.compute_cosine_similarity(neutral_latent, middle_latent)

        if neutral_latent is not None and older_latent is not None:
            metrics["cosine_latent_neutral_older"] = self.compute_cosine_similarity(neutral_latent, older_latent)

        return metrics, images

    def evaluate_and_save_results(self, quadruplet: tuple, seed: int = None, output_dir: str = "generated_images") -> pd.DataFrame:
        """
        Generates a DataFrame with metrics and also returns the generated images.
        """
        import pandas as pd

        metrics, images = self.evaluate_prompt_quadruplet_denoising_space(quadruplet, seed=seed, output_dir=output_dir)

        results = []
        prompts = ["neutral", "young", "middle", "older"]

        for i, prompt in enumerate(prompts):
            results.append({
                "Quadruplet": quadruplet[i],
                "Prompt": prompt,
                "Cosine_Similarity_Prompt": self.safe_round(metrics.get(f"cosine_prompt_neutral_{prompt}", "N/A")),
                "Cosine_Similarity_Image": self.safe_round(metrics.get(f"cosine_image_neutral_{prompt}", "N/A")),
                "Cosine_Similarity_Latent": self.safe_round(metrics.get(f"cosine_latent_neutral_{prompt}", "N/A")),
                "ResNet_Similarity": self.safe_round(metrics.get(f"resnet_similarity_neutral_{prompt}", "N/A")),
                "Dino16_Similarity": self.safe_round(metrics.get(f"cosine_dino_s16_neutral_{prompt}", "N/A")),
                "Split_Product": self.safe_round(metrics.get(f"split_product_{prompt}", "N/A")),
            })

        df = pd.DataFrame(results)
        return df, images

    def safe_round(self, value):
        """
        Rounds the value if numeric, otherwise returns the value as is.
        """
        if isinstance(value, (int, float)):
            return round(value, 3)
        return value

def plot_images_with_metrics(images: dict, metrics_df: pd.DataFrame):
    fig, axes = plt.subplots(1, len(images), figsize=(20, 10))

    for ax, (prompt, image) in zip(axes, images.items()):
        ax.imshow(image)
        ax.axis("off")

        metric_row = metrics_df.loc[metrics_df["Prompt"] == prompt]

        if metric_row.empty:
            metric = "N/A"
        else:
            prompt_similarity = metric_row["Cosine_Similarity_Prompt"].values[0] if not pd.isna(metric_row["Cosine_Similarity_Prompt"].values[0]) else "N/A"
            image_similarity = metric_row["Cosine_Similarity_Image"].values[0] if not pd.isna(metric_row["Cosine_Similarity_Image"].values[0]) else "N/A"
            latent_similarity = metric_row["Cosine_Similarity_Latent"].values[0] if not pd.isna(metric_row["Cosine_Similarity_Latent"].values[0]) else "N/A"
            resnet_similarity = metric_row["ResNet_Similarity"].values[0] if not pd.isna(metric_row["ResNet_Similarity"].values[0]) else "N/A"
            dino16_similarity = metric_row["Dino16_Similarity"].values[0] if not pd.isna(metric_row["Dino16_Similarity"].values[0]) else "N/A"
            split_product = metric_row["Split_Product"].values[0] if not pd.isna(metric_row["Split_Product"].values[0]) else "N/A"

            if prompt != 'neutral':
                ax.set_title(
                    f"{prompt.capitalize()}\n"
                    f"CLIP Prompt: {prompt_similarity if prompt_similarity != 'N/A' else 'N/A'}\n"
                    f"CLIP Image: {image_similarity if image_similarity != 'N/A' else 'N/A'}\n"
                    f"VAE Latent: {latent_similarity if latent_similarity != 'N/A' else 'N/A'}\n"
                    f"ResNet: {resnet_similarity if resnet_similarity != 'N/A' else 'N/A'}\n"
                    f"DINO-s16: {dino16_similarity if dino16_similarity != 'N/A' else 'N/A'}\n"
                    f"Split Product: {split_product if split_product != 'N/A' else 'N/A'}"
                )
            else:
                ax.set_title(
                    f"{prompt.capitalize()}\n"
                    f"Prompt: {prompt_similarity if prompt_similarity != 'N/A' else 'N/A'}\n"
                    f"Image: {image_similarity if image_similarity != 'N/A' else 'N/A'}\n"
                    f"Latent: {latent_similarity if latent_similarity != 'N/A' else 'N/A'}"
                )

    plt.tight_layout()
    plt.show()

def generate_quadruplets_for_activities(activities):
    quadruplets = []

    for activity in activities:
        activity = activity.lower()
        neutral = f"An ultra realistic portrait photo of a person {activity}"
        young = f"An ultra realistic portrait photo of a 25 years-old young person {activity}"
        middle = f"An ultra realistic portrait photo of a 45 year-old person middle-aged person {activity}"
        older = f"An ultra realistic portrait photo of a 75 years-old older person {activity}"

        quadruplets.append((neutral, young, middle, older))

    return quadruplets

def main():
    # Load Stable Diffusion XL pipeline
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float32,
        use_safetensors=True,
        cache_dir="C:/Users/David/.cache/huggingface/hub",
    ).to("cuda")

    pipe.enable_xformers_memory_efficient_attention()
    pipe.enable_attention_slicing("auto")
    pipe.enable_sequential_cpu_offload()

    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda")
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    # Initialize evaluator
    evaluator = Evaluator(pipe, clip_model, clip_processor, scheduler_name="euler_ancestral")

    activities = activitats
    
    quadruplets = generate_quadruplets_for_activities(activities)

    all_results = []

    for quadruplet in quadruplets:
        print(f"Evaluating quadruplet: {quadruplet}")
        df, images = evaluator.evaluate_and_save_results(quadruplet, seed=123, output_dir="generated_images/500p_quadruplets_v1")
        df_filtered = df[df['Prompt'] != 'neutral']
        all_results.append(df_filtered)
        print("Denoising space evaluation results:", df)
        plot_images_with_metrics(images, df)

    final_df = pd.concat(all_results, ignore_index=True)
    return final_df

if __name__ == "__main__":
    results_df = main()

In [ ]:
results_df.to_csv("quadruplets_results.csv", index=False)

In [ ]:
def calculate_metrics_table(df):
    metrics = {
        'CLIP': ['Cosine_Similarity_Prompt'],
        'UNET-VAE': ['Cosine_Similarity_Latent'],
        'Resnet': ['ResNet_Similarity'],
        'CLIP_img': ['Cosine_Similarity_Image'],
        'DINO': ['Dino16_Similarity'],
        'SPLIT': ['Split_Product']
    }

    results = pd.DataFrame(index=['Prompt embeddings', 'Pre-Image embeddings', 'Image embeddings',
                                'Image embeddings', 'Image embeddings', 'Image embeddings'],
                          columns=['Metric', 'Modulo', 'Neutral vs Young', 'Neutral vs Middle', 'Neutral vs Older', 'Chi2'])

    results['Metric'] = ['Cosine similarity', 'Cosine similarity', 'Cosine similarity',
                        'Cosine similarity', 'Cosine similarity', 'Split Product']
    results['Modulo'] = ['CLIP', 'UNET-VAE', 'Resnet', 'CLIP', 'DINO', 'SPLIT']

    # Calculate means for each group
    for i, (module, metric_list) in enumerate(metrics.items()):
        for metric in metric_list:
            young_mean = df[df['Prompt'] == 'young'][metric].mean()
            middle_mean = df[df['Prompt'] == 'middle'][metric].mean()
            older_mean = df[df['Prompt'] == 'older'][metric].mean()

            results.iloc[i, results.columns.get_loc('Neutral vs Young')] = round(young_mean, 3)
            results.iloc[i, results.columns.get_loc('Neutral vs Middle')] = round(middle_mean, 3)
            results.iloc[i, results.columns.get_loc('Neutral vs Older')] = round(older_mean, 3)

    return results

metrics_table = calculate_metrics_table(results_df)
print("\nMetrics table:")
print(metrics_table)

metrics_table.to_csv("generated_images/90p_quadruplets/quadruplets_results.csv")